# MFA-Modell für Roggenstroh - Modulare Version

## Changelog / Änderungsprotokoll

* **Version:** 1.0
* **Datum:** 21.06.2025
* **Autor:** [Johannes Scholz, Lukas Hoppe]
* **Änderungen:**
    * Grundstruktur für modulares Notebook erstellt.
    * Code in Funktionsblöcke unterteilt (Setup, Berechnungen, Visualisierung).

# Section 0: Importing the packages & basic settings translator

In [ ]:
### Load packages ###

# Load general libraries
import sys, os
import numpy as np
import pandas as pd
from scipy.stats import lognorm
import xlsxwriter
import matplotlib.pyplot as plt
from matplotlib.ticker import (MultipleLocator,
                               FormatStrFormatter,
                               AutoMinorLocator)
import warnings
import re
from collections import defaultdict
from scipy.optimize import minimize
import copy

# Load ODYM package
# Add ODYM module directory to system path, absolute
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())),'framework', 'ODYM-master_20241127', 'odym', 'modules')) 

# Import the ODYM class file
import ODYM_Classes as msc 
# Import the ODYM function file
import ODYM_Functions as msf
# Import the dynamic stock model library
import dynamic_stock_model as dsm 


# Load bioDYM_addon
# Add ODYM module directory to system path, absolute
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), 'framework', 'bioDYM_add-on', 'modules')) 

# Import classes for first order model process
import bioDYM_classes as bicl
# Import plotting functions
import bioDYM_plotting as bipl
# Import export functions
import bioDYM_export as bix




# Enables plotting directly in the notebook (in most cases this is already enabled)
%matplotlib inline

# Section 1: Configuration

In [2]:
# ====================================================================
# Section 1: Model Configuration
# All user-defined settings for a scenario run go here.
# ====================================================================

# --- File Paths ---
EXCEL_FILE_PATH = '250625_Template_CS0.xlsx'

# --- Model Scope ---
START_YEAR = 2025
END_YEAR = 2050
ELEMENTS = ['material', 'WC', 'DM', 'CC'] # Elements to be tracked

# In Section 1: Model Configuration

# --- Model Calculation Switches ---
# Set to True to enable the calculation, False to disable.
# This allows for simpler test runs without DSM or FOMP.
RUN_DSM_CALCULATION = True
RUN_FOMP_CALCULATION = True


In [3]:
# ===================================================================
# SECTION 1.1: MONTE CARLO & UNCERTAINTY CONFIGURATION
# ===================================================================

# Switch to activate the Monte Carlo simulation
RUN_MONTE_CARLO = False  # Set to False for a single, deterministic run

# Number of iterations for the simulation
MC_ITERATIONS = 100     # Start with a lower number like 100 for testing

# Definition of uncertain parameters
# Format: { 'parameter_name': {'distribution': 'type', 'param1': value1, 'param2': value2, ...} }
UNCERTAINTY_PARAMS = {
    # Example 1: A Transfer Coefficient with a uniform distribution
    'TC_03_04': {'distribution': 'uniform', 'min': 0.4, 'max': 0.6},
    
    # Example 2: A DSM lifetime with a triangular distribution (expert guess)
    'dsm_6_lifetimes_Mean_0': {'distribution': 'triangular', 'min': 25, 'mode': 30, 'max': 40}, # Reifen
    
    # Example 3: A FOMP decay rate with a normal distribution
    'fomp_8_k1': {'distribution': 'normal', 'mean': 0.025, 'std': 0.005}
}

# Section 2: Function definitions

### Erklärung der Berechnungs-Engine: Der iterative Solver

Die Berechnung eines komplexen Stoffstromsystems, insbesondere mit Lagern, zeitlichen Verzögerungen und Kreisläufen, erfordert eine robuste Methode, die sicherstellt, dass alle Flüsse in der korrekten Reihenfolge berechnet werden.

#### Das Problem: Das "Wasserfall-Prinzip"

Ein einfacher, rein sequenzieller Ansatz (erst alle TCs, dann alle DSMs, etc.) funktioniert wie ein Wasserfall – die Berechnung fließt nur in eine Richtung. Dies scheitert, wenn es im System Kreisläufe oder komplexe Abhängigkeiten gibt. Ein Prozess `C` muss dann vielleicht auf das Ergebnis eines Prozesses `B` warten, der seinerseits auf `A` wartet. Dieser starre Ablauf kann zu Berechnungsfehlern oder "stillen Fehlern" führen, bei denen falsche Zwischenergebnisse verwendet werden.

#### Die Lösung: Ein "iterativer Solver"

Die hier implementierte Methode ist ein **iterativer Solver**. Anstatt zu versuchen, die "richtige" Reihenfolge vorab zu erraten, dreht die Rechen-Engine so lange Runden ("Iterationen"), bis sich im gesamten System nichts mehr ändert und es einen stabilen Zustand erreicht hat.

**Der Ablauf in jeder einzelnen Iteration:**

1.  **Berechnung der "einfachen" TC-Flüsse:** Der Solver versucht zuerst, alle normalen, TC-basierten Flüsse zu berechnen. **Wichtig:** Er überspringt dabei aber konsequent alle Flüsse, deren Startprozess als "Spezialprozess" (DSM oder FOMP) definiert ist. Damit wird verhindert, dass die Ergebnisse der Spezial-Logik vorzeitig mit einer falschen TC-Berechnung überschrieben werden.

2.  **Ausführung der Spezial-Modelle (DSM/FOMP):** Als Nächstes prüft der Solver für jeden Spezialprozess, ob alle seine Zuflüsse mittlerweile bekannt sind.
    * Wenn ja, wird die entsprechende Spezialfunktion (`calculate_dynamic_stock` oder `calculate_fomp`) ausgeführt, die den korrekten, physikalisch basierten Abfluss berechnet.
    * Wenn nein, wartet der Solver bis zur nächsten Runde.

3.  **Wiederholung und Konvergenz:** Dieser gesamte Prozess (Schritt 1 & 2) wird wiederholt. In der nächsten Runde sind die Abflüsse der Spezialmodelle aus der vorherigen Runde nun bekannte Zuflüsse für andere Prozesse. Dadurch können in der TC-Berechnung weitere, bisher unlösbare Flüsse berechnet werden. Die Information "frisst" sich so lange durch das System, bis alle Flüsse bekannt sind.

4.  **Stabilitäts-Check:** Wenn die Engine eine komplette Runde durchläuft, ohne einen einzigen neuen Wert berechnen zu können, bedeutet das, das System ist stabil (konvergiert). Die Schleife bricht dann ab, um unnötige Rechenzeit zu sparen.

Dieser iterative Ansatz ist die Standardmethode zur Lösung komplexer Stoffstrom-Systeme, da er Abhängigkeiten und Kreisläufe robust und automatisch auflöst.

In [4]:
# TEMPORÄRE DEBUG-ZELLE: Spaltennamen überprüfen
import pandas as pd

# Bitte stellen Sie sicher, dass der Dateiname korrekt ist
excel_file_path = '250625_Template_CS0.xlsx' 

df_check = pd.read_excel(excel_file_path, sheet_name='2_3_Process_TCs')

print("Gefundene Spalten im Blatt '2_3_Process_TCs':")
print(list(df_check.columns))

Gefundene Spalten im Blatt '2_3_Process_TCs':
['Spalte1', 'ID', 'Process_ID', 'Name(EN)', 'Region', 'Carbon_Stock', 'Life_Phase', 'Description', 'Process_Type', 'TC?', 'Dyn_TC?', 'Stock?', 'Initial_Stock?', 'DSM?', 'FOMP?', 'Nr. Outflows?', 'Output_Flow', 'Flow_ID', 'TC_ID', 'TC_Value', 'Titel', 'Year publication', 'Author', 'Type of the Study', 'URL']


c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


### 2.1: Define Model Scope & Classifications

### 2.2: Initialize the main MFA System object

### 2.3 validate_input_data

### 2.3 load_and_define_processes

In [8]:
# ===================================================================
# FINAL CORRECTED FUNCTION
# ===================================================================
def load_and_define_processes(mfa_system, excel_path):
    """
    This function now ONLY defines the structure of processes and stocks,
    without setting any initial values.
    """
    print("--> Defining process and stock structures...")
    
    input_data = pd.read_excel(excel_path, sheet_name=None, header=0, engine='openpyxl', na_values=['N.A.', 'NA', 'n/a'])
    validate_input_data(input_data) # Validation remains important

    process_definitions = input_data['2_1_Definition_Processes']
    for index, row in process_definitions.iterrows():
        if pd.notna(row['Name(EN)']):
            process_id = int(row['ID'])
            has_tcs = 'TC' if 'TC?' in row and row['TC?'] == 'Yes' else 'None'
            mfa_system.ProcessList.append(msc.Process(Name=row['Name(EN)'], ID=process_id, Extensions=has_tcs))
            
            # Create stock objects if needed
            if 'Stock?' in row and row['Stock?'] == 'Yes':
                mfa_system.StockDict[f"dS_{process_id}"] = msc.Stock(Name=f"dS_{process_id}", P_Res=process_id, Type=1, Indices='t,e')
                mfa_system.StockDict[f"S_{process_id}"] = msc.Stock(Name=f"S_{process_id}", P_Res=process_id, Type=0, Indices='t,e')

    # Values will be set in the next function
    return mfa_system, input_data

In [9]:
# ===================================================================
# NEUE KORRIGIERTE FUNKTION zum Laden der DSM-Parameter
# ===================================================================

def load_dsm_parameters(excel_data):
    """
    Liest das Blatt '3_1_Definition_DSM' und erstellt das DSM_PARAMS Dictionary.
    
    BUGFIX: Diese Version wandelt die Spalte 'Process_ID' explizit in den
    Datentyp Integer um. Dies behebt den Fehler, bei dem Prozess-IDs als
    Float (z.B. 7.0) statt als Integer (z.B. 7) eingelesen wurden, was zu
    fehlgeschlagenen Zugriffen auf die Lagerobjekte führte.
    """
    sheet_name = '3_1_Definition_DSM'
    print(f"--> Loading DSM parameters from sheet '{sheet_name}'...")
    
    if sheet_name not in excel_data:
        print(f"--> INFO: Sheet '{sheet_name}' not found. Using empty DSM configuration.")
        return {}

    df_dsm = excel_data[sheet_name]
    
    # --- START DES BUGFIX ---
    # Prüfen, ob die Spalte existiert, um Folgefehler zu vermeiden.
    if 'Process_ID' not in df_dsm.columns:
        print(f"--> FATAL ERROR: Spalte 'Process_ID' nicht im Blatt '{sheet_name}' gefunden.")
        return {}
        
    # Entferne Zeilen ohne Prozess_ID und erzwinge den Datentyp Integer.
    # Dies ist der entscheidende Schritt, um den 7 vs 7.0 Fehler zu beheben.
    df_dsm = df_dsm.dropna(subset=['Process_ID'])
    df_dsm['Process_ID'] = df_dsm['Process_ID'].astype(int)
    # --- ENDE DES BUGFIX ---
    
    dsm_params = {}

    # Gruppiere nach der nun korrekten Integer-ID
    for process_id, group in df_dsm.groupby('Process_ID'):
        # 'process_id' ist jetzt garantiert ein Integer
        group = group.sort_values(by='Category_ID')
        
        dsm_params[process_id] = {
            'inflow_split': list(group['Inflow_Split_[%]']),
            'lifetimes': {
                'Type': list(group['Lifetime_Type'])[0],
                'Mean': list(group['Lifetime_Mean']),
                'StdDev': list(group['Lifetime_StdDev'])
            },
            'category_names': list(group['Category_Name'])
        }
        
    print(f"--> Successfully loaded configurations for {len(dsm_params)} DSM process(es).")
    return dsm_params

In [10]:
def load_fomp_parameters(excel_data):
    """
    Reads the '3_2_Definition_FOMP' sheet and constructs the FOMP_PARAMS dictionary.
    This version ignores empty rows.
    """
    # ÄNDERUNG: Korrekter Blattname
    sheet_name = '3_2_Definition_FOMP' 
    print(f"--> Loading FOMP parameters from sheet '{sheet_name}'...")

    if sheet_name not in excel_data:
        print(f"--> INFO: Sheet '{sheet_name}' not found. Using empty FOMP configuration.")
        return {}
        
    df_fomp = excel_data[sheet_name]
    fomp_params = {}

    for _, row in df_fomp.iterrows():
        # ÄNDERUNG: Prüfen, ob die Zeile leer ist, bevor wir sie verarbeiten
        if pd.isna(row['Process_ID']):
            continue # Überspringe diese Zeile und gehe zur nächsten

        process_id = int(row['Process_ID'])
        param_name = row['Parameter_Name']
        value = row['Value']
        
        if process_id not in fomp_params:
            fomp_params[process_id] = {}
        
        try:
            fomp_params[process_id][param_name] = float(value)
        except (ValueError, TypeError):
            fomp_params[process_id][param_name] = value
            
    print(f"--> Successfully loaded configurations for {len(fomp_params)} FOMP process(es).")
    return fomp_params

In [11]:
# ===================================================================
# ERSETZTE LADEFUNKTION FÜR UNSICHERHEITEN
# ===================================================================
def load_uncertainty_definitions(excel_data):
    """
    Reads the '3_Uncertainty_Parameters' sheet with the new, self-documenting
    structure and converts it into the UNCERTAINTY_PARAMS dictionary format.
    """
    sheet_name = '4_1_Uncertainty_Parameters'
    print(f"--> Loading uncertainty definitions from sheet '{sheet_name}'...")
    
    if sheet_name not in excel_data:
        print(f"--> INFO: Sheet '{sheet_name}' not found. No uncertainties will be loaded.")
        return {}

    df_uncertainty = excel_data[sheet_name].dropna(subset=['Parameter_Name'])
    uncertainty_params = {}

    for _, row in df_uncertainty.iterrows():
        param_name = row['Parameter_Name']
        dist_type = row['Distribution']
        definition = {'distribution': dist_type}
        
        # Read values only if they are not empty (NaN)
        if dist_type == 'uniform':
            if pd.notna(row['Min']) and pd.notna(row['Max']):
                definition['min'] = row['Min']
                definition['max'] = row['Max']
        elif dist_type == 'normal':
            if pd.notna(row['Mean']) and pd.notna(row['StdDev']):
                definition['mean'] = row['Mean']
                definition['std'] = row['StdDev']
        elif dist_type == 'triangular':
            if pd.notna(row['Min']) and pd.notna(row['Mode']) and pd.notna(row['Max']):
                definition['min'] = row['Min']
                definition['mode'] = row['Mode']
                definition['max'] = row['Max']
        
        if len(definition) > 1: # Add only if parameters were found
            uncertainty_params[param_name] = definition
        
    print(f"--> Successfully loaded {len(uncertainty_params)} uncertainty parameter definition(s).")
    return uncertainty_params

### 2.11 sample_parameters

In [18]:
# ===================================================================
# NEW FUNCTION FOR MONTE CARLO SAMPLING
# ===================================================================
def sample_parameters(uncertainty_defs):
    """
    Draws a new random value for each defined uncertain parameter.
    Returns a dictionary with the new values.
    """
    sampled_values = {}
    for param_name, definition in uncertainty_defs.items():
        dist_type = definition.get('distribution')

        if dist_type == 'uniform':
            sampled_values[param_name] = np.random.uniform(definition['min'], definition['max'])
        elif dist_type == 'normal':
            sampled_values[param_name] = np.random.normal(definition['mean'], definition['std'])
        elif dist_type == 'triangular':
            sampled_values[param_name] = np.random.triangular(definition['min'], definition['mode'], definition['max'])
        elif dist_type == 'lognormal':
             sampled_values[param_name] = np.random.lognormal(definition['mean'], definition['std'])
        else:
            print(f"WARNING: Unknown distribution type '{dist_type}' for parameter '{param_name}'. Parameter will not be sampled.")
            
    return sampled_values

# Section 3: Main workflow (execution)

In [21]:
# ===================================================================
# Section 3: Main Execution (Final Version with Monte Carlo)
# ===================================================================

# --- System-Setup (bleibt gleich) ---
model_classification, index_table = define_model_scope(START_YEAR, END_YEAR, ELEMENTS)
my_mfa_system_base = initialize_mfa_system(model_classification, index_table)
my_mfa_system_base, all_excel_data = load_and_define_processes(my_mfa_system_base, EXCEL_FILE_PATH)

DSM_PARAMS = load_dsm_parameters(all_excel_data)
FOMP_PARAMS = load_fomp_parameters(all_excel_data)
UNCERTAINTY_PARAMS = load_uncertainty_definitions(all_excel_data)

# Die Konfiguration wird nicht mehr hier aufgerufen, sondern innerhalb des Loops
# my_mfa_system_configured, _ = define_flows_and_parameters(my_mfa_system_base, all_excel_data, DSM_PARAMS, FOMP_PARAMS)

# --- Initialisiere Ergebnis-Variablen ---
df_mc_results = None
my_mfa_system_with_results = None
dsm_details = None

# --- ENTSCHEIDUNG UND BERECHNUNG ---
if RUN_MONTE_CARLO:
    print(f"\n--- STARTING MONTE CARLO SIMULATION ({MC_ITERATIONS} iterations) ---")
    mc_run_results = []
    
    # Fortschrittsbalken für eine bessere Übersicht
    try:
        from tqdm.notebook import tqdm
        iterator = tqdm(range(MC_ITERATIONS), desc='MC Runs')
    except ImportError:
        iterator = range(MC_ITERATIONS)

    for i in iterator:
        # 1. Neue Parameterwerte für diese Iteration sampeln
        sampled_values = sample_parameters(UNCERTAINTY_PARAMS)
        
        # 2. Temporäre Kopien der Parameter-Sets erstellen, um die Originale nicht zu verändern
        temp_dsm_params = copy.deepcopy(DSM_PARAMS)
        temp_fomp_params = copy.deepcopy(FOMP_PARAMS)
        tc_updates = {}

        # 3. Gesampelte Werte den richtigen Parameter-Typen zuordnen
        for name, value in sampled_values.items():
            if name.startswith('TC_'):
                tc_updates[name] = value
            elif name.startswith('fomp_'):
                try:
                    parts = name.split('_'); pid = int(parts[1]); param_key = parts[2]
                    if pid in temp_fomp_params: temp_fomp_params[pid][param_key] = value
                except (IndexError, ValueError): pass
            elif name.startswith('dsm_'):
                try:
                    parts = name.split('_'); pid = int(parts[1]); dict_key = parts[2]; param_key = parts[3]; list_index = int(parts[4])
                    if pid in temp_dsm_params and list_index < len(temp_dsm_params[pid][dict_key][param_key]):
                        temp_dsm_params[pid][dict_key][param_key][list_index] = value
                except (IndexError, ValueError): pass

        # 4. MFA-Berechnung mit den für diesen Lauf modifizierten Parametern ausführen
        run_results, _ = run_mfa_calculation(temp_dsm_params, temp_fomp_params, tc_updates=tc_updates)
        
        # 5. Schlüsselergebnisse (KPIs) extrahieren und speichern
        if run_results:
            # Beispiel-KPI: Finaler C-Lagerbestand im Boden (Prozess 8, Element 'CC')
            final_c_stock_soil = run_results.StockDict['S_8'].Values[-1, 3] # S_8, letztes Jahr, 4. Element ('CC')
            
            current_run_data = sampled_values.copy()
            current_run_data['run_id'] = i
            current_run_data['final_C_stock_soil'] = final_c_stock_soil
            mc_run_results.append(current_run_data)

    # 6. Alle Ergebnisse in einem DataFrame zusammenfassen
    df_mc_results = pd.DataFrame(mc_run_results)
    my_mfa_system_with_results = None
    print("\n--- MONTE CARLO SIMULATION COMPLETE ---")

else:
    # --- DETERMINISTISCHER EINZELLAUF ---
    print("\n--- STARTING SINGLE DETERMINISTIC RUN ---")
    my_mfa_system_with_results, dsm_details = run_mfa_calculation(DSM_PARAMS, FOMP_PARAMS)
    print("\nCalculation complete.")

--> Model scope and classifications defined.
--> MFA system object initialized.
--> Defining process and stock structures...


c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  

--> Validating input data structure...
--> Input data validation successful. All required sheets and columns are present.
--> Loading DSM parameters from sheet '3_1_Definition_DSM'...
--> Successfully loaded configurations for 2 DSM process(es).
--> Loading FOMP parameters from sheet '3_2_Definition_FOMP'...
--> Successfully loaded configurations for 1 FOMP process(es).
--> Loading uncertainty definitions from sheet '4_1_Uncertainty_Parameters'...
--> Successfully loaded 4 uncertainty parameter definition(s).

--- STARTING SINGLE DETERMINISTIC RUN ---
--> MFA system object initialized.
--> Defining process and stock structures...


c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\Johannes\anaconda3\envs\biODYM_anaconda_environment\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  

--> Validating input data structure...
--> Input data validation successful. All required sheets and columns are present.
--> Defining flows, parameters, and setting all initial values...
--> All stocks and flows initialized to zero.
--> Populated data for primary input flows.
--> Generating dynamic TC time series via interpolation...
--> Generated 12 dynamic TC parameter(s).
--> Defined 56 parameters in total.
--> Calculating FOMP outflows...
    ... calculating outflow from FOMP process 8
--> FOMP outflow calculation finished.
--> Calculating final stock balances for ALL processes...
--> Stock balance calculation finished.

Calculation complete.


### Überprüfung der Massenbilanz

Die folgende Grafik ist das wichtigste Werkzeug zur Überprüfung der Modellkonsistenz. Sie zeigt den Massenbilanzfehler für jeden Prozess, berechnet nach der Formel:

**`Fehler = Σ Zuflüsse - Σ Abflüsse - Lagerveränderung (dS)`**

**Wie man die Grafik liest:**
* **Perfektes Gleichgewicht:** Ein Prozess ist perfekt bilanziert, wenn sein Balken genau auf der Nulllinie liegt.
* **Positiver Fehler (Balken > 0):** Es wurde mehr Masse "erschaffen" als im System sein dürfte. Mögliche Ursache: Ein Abfluss oder eine Lagerbildung fehlt in der Definition.
* **Negativer Fehler (Balken < 0):** Es ist Masse "verschwunden". Mögliche Ursache: Ein Zufluss fehlt oder ein Abfluss wird doppelt gezählt.

Je größer die Abweichung von Null, desto gravierender ist der Fehler in der Modelllogik für diesen Prozess.

# Section 4: Results and visualization

In [22]:
# ===================================================================
# Section 4 - DEBUG PROBE: Inspecting the final plot data
# ===================================================================

print("\n" + "="*40)
print("FINAL DEBUG PROBE: Inspecting the 'dsm_details' dictionary before plotting")
print("="*40)

if 'dsm_details' in locals() and dsm_details is not None:
    # Prozess 7 ist unser Testfall mit einem Initial Stock von 2000 Mg
    process_id_to_check = 7
    if process_id_to_check in dsm_details:
        details_for_p7 = dsm_details[process_id_to_check]
        
        print(f"\n--- Data for Process {process_id_to_check} ---")
        
        # Wir prüfen das Array, das für den Plot des Initial Stocks verwendet wird
        initial_stock_plot_data = details_for_p7.get('initial_stock_ts')
        
        if initial_stock_plot_data is not None:
            print(f"Shape of 'initial_stock_ts' array: {initial_stock_plot_data.shape}")
            print(f"First 3 values of the 'material' component for the plot: {initial_stock_plot_data[:3, 0]}")
            
            if np.sum(initial_stock_plot_data) == 0:
                print("\n>>> DIAGNOSE: Die an den Plot übergebenen Daten sind komplett Null. Der Fehler liegt in der BERECHNUNGS-Logik.")
            else:
                print("\n>>> DIAGNOSE: Die an den Plot übergebenen Daten enthalten Werte > 0. Der Fehler liegt in der VISUALISIERUNGS-Logik.")
        else:
            print("FEHLER: Der Schlüssel 'initial_stock_ts' wurde in dsm_details für Prozess 7 nicht gefunden.")
    else:
        print(f"FEHLER: Prozess {process_id_to_check} wurde nicht im dsm_details-Dictionary gefunden.")
else:
    print("FEHLER: Die Variable 'dsm_details' existiert nicht oder ist leer. Es wurden keine DSM-Ergebnisse zurückgegeben.")

print("="*40)


FINAL DEBUG PROBE: Inspecting the 'dsm_details' dictionary before plotting

--- Data for Process 7 ---
Shape of 'initial_stock_ts' array: (26, 4)
First 3 values of the 'material' component for the plot: [2000.         1936.84210526 1875.67867036]

>>> DIAGNOSE: Die an den Plot übergebenen Daten enthalten Werte > 0. Der Fehler liegt in der VISUALISIERUNGS-Logik.


# ====================================================================
# Section 4: Results & Visualization
# ====================================================================

### Überprüfung der Massenbilanz

Die folgende Grafik ist das wichtigste Werkzeug zur Überprüfung der Modellkonsistenz. Sie zeigt den Massenbilanzfehler für jeden Prozess, berechnet nach der Formel:

**`Fehler = Σ Zuflüsse - Σ Abflüsse - Lagerveränderung (dS)`**

**Wie man die Grafik liest:**
* **Perfektes Gleichgewicht:** Ein Prozess ist perfekt bilanziert, wenn sein Balken genau auf der Nulllinie liegt.
* **Positiver Fehler (Balken > 0):** Es wurde mehr Masse "erschaffen" als im System sein dürfte. Mögliche Ursache: Ein Abfluss oder eine Lagerbildung fehlt in der Definition.
* **Negativer Fehler (Balken < 0):** Es ist Masse "verschwunden". Mögliche Ursache: Ein Zufluss fehlt oder ein Abfluss wird doppelt gezählt.

In [28]:
# ===================================================================
# Section 4: Results and Visualization (Final Corrected Version)
# ===================================================================

# Prüfe zuerst, welcher Modus gelaufen ist
if RUN_MONTE_CARLO:
    # --- VISUALIZATION FOR MONTE CARLO RUN ---
    # Stelle sicher, dass Ergebnisse vorhanden sind
    if df_mc_results is not None and not df_mc_results.empty:
        print("\n--- Monte Carlo Results Summary ---")
        print(df_mc_results.describe())

        # Zeige die Verteilung des wichtigsten Ergebnisses
        plot_mc_distribution(df_mc_results, 'final_C_stock_soil', unit='Mg C')

        # Zeige eine Beispiel-Sensitivitätsanalyse
        uncertain_input_to_test = 'fomp_8_k1'
        if uncertain_input_to_test in df_mc_results.columns:
            plot_mc_sensitivity_scatter(df_mc_results, uncertain_input_to_test, 'final_C_stock_soil', unit='Mg C')
        else:
            print(f"\nNOTE: Sensitivity plot for '{uncertain_input_to_test}' skipped, as it was not in the uncertainty parameters.")
    else:
        print("\nINFO: Monte Carlo run was selected, but no results were generated.")
        
else:
    # --- VISUALIZATION FOR SINGLE DETERMINISTIC RUN ---
    # Stelle sicher, dass Ergebnisse vorhanden sind
    if my_mfa_system_with_results is not None:
        print("\n--- Displaying results for single run ---")
        
        # --- 4.1 Interactive Sankey Diagram ---
        print("Displaying interactive Sankey diagram:")
        # Dieser Aufruf ist jetzt sicher, da er nur im richtigen "Pfad" stattfindet
        plot_interactive_sankey(my_mfa_system_with_results)
        
        # --- 4.2 Process Dynamics Plot (Inflow-Stock-Outflow) ---
        print("\nDisplaying Inflow-Stock-Outflow Dynamics:")
        plot_process_dynamics(my_mfa_system_with_results, all_excel_data['2_1_Definition_Processes'])
        
        # --- 4.3 Dynamic Stock Composition Plot ---
        print("\nDisplaying Dynamic Stock Composition:")
        if dsm_details:
             plot_dynamic_stock_composition(dsm_details, my_mfa_system_with_results)
        else:
             print("--> DSM calculation was not run or no DSM processes are defined. Nothing to display.")
        
        # --- 4.4 FOMP Dynamics Plot ---
        print("\nDisplaying FOMP Dynamics:")
        if FOMP_PARAMS:
             plot_fomp_dynamics(my_mfa_system_with_results, FOMP_PARAMS)
        else:
             print("--> No FOMP parameters defined. Nothing to display.")
        
        # --- 4.5 Flow Dynamics Plot ---
        print("\nDisplaying Flow Dynamics:")
        plot_flow_dynamics(my_mfa_system_with_results)
    else:
        print("\nINFO: Single run was selected, but no results were generated.")


--- Displaying results for single run ---
Displaying interactive Sankey diagram:


interactive(children=(IntSlider(value=2025, description='Year', max=2050, min=2025), Dropdown(description='Ele…

FigureWidget({
    'data': [{'link': {'source': [0, 1, 2, 3, 3, 4, 4, 5, 6, 7, 7, 8, 9, 9],
                       'target': [2, 2, 3, 4, 5, 0, 1, 6, 9, 0, 1, 0, 8, 7],
                       'value': [100.0, 100.0, 200.0, 100.0, 100.0, 50.0, 50.0,
                                 100.0, 53.414960640068124, 63.19769925769572, 0.0,
                                 30.792318331808985, 21.36598425602725,
                                 32.04897638404087]},
              'node': {'color': 'blue',
                       'label': [Atmosphere, Environment, Cultivation, Harvest,
                                 Food, Straw d&C, Use_Straw_Roof,
                                 Incineration_Roof, Composting_roof, Roof_EoL]},
              'type': 'sankey',
              'uid': '2879add0-803d-4da9-9213-0cbb92e38b85'}],
    'layout': {'font': {'size': 12},
               'height': 700,
               'margin': {'b': 20, 'l': 10, 'r': 10, 't': 50},
               'template': '...',
               


Displaying Inflow-Stock-Outflow Dynamics:


interactive(children=(Dropdown(description='Process:', options=('Atmosphere', 'Environment', 'Use_Straw_Roof',…

...

interactive(children=(Dropdown(description='Process:', options=('Composting_roof',), value='Composting_roof'),…

FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'Inflow',
              'type': 'scatter',
              'uid': '97d13237-09ca-4250-a765-8adab6b22d4a',
              'x': [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
                    2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
                    2045, 2046, 2047, 2048, 2049, 2050],
              'xaxis': 'x',
              'y': array([21.36598426, 20.24489727, 19.19070299, 18.20230739, 17.27954307,
                          16.42344676, 15.63675735, 14.92924059, 14.38450652, 14.40342197,
                          15.43152529, 16.72023112, 17.72164287, 18.66641031, 20.01082727,
                          21.80306498, 23.94457433, 26.26300572, 28.66266014, 30.96854021,
                          32.39082319, 32.5362709 , 33.6454829 , 35.88864593, 36.97399669,
                          37.2104085 ]),
              'yaxis': 'y'},
             {'mode': 'lines',
              'name': 'Stoc


Displaying Flow Dynamics:


interactive(children=(SelectMultiple(description='Flows:', index=(0,), options=('F_00_02', 'F_01_02', 'F_02_03…

FigureWidget({
    'data': [{'mode': 'lines',
              'name': 'F_00_02',
              'type': 'scatter',
              'uid': '0b774253-caa1-42a6-a7ac-3247cba4c2e2',
              'x': [2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032, 2033, 2034,
                    2035, 2036, 2037, 2038, 2039, 2040, 2041, 2042, 2043, 2044,
                    2045, 2046, 2047, 2048, 2049, 2050],
              'y': array([100., 110., 120., 130., 140., 150., 160., 170., 180., 190., 200.,  10.,
                          220., 230., 240., 250., 260., 270., 280., 290., 300., 310., 320., 330.,
                          340., 350.])}],
    'layout': {'barmode': 'overlay',
               'hovermode': 'x unified',
               'template': '...',
               'title': {'text': 'Time Series for Selected Flows (MATERIAL)'},
               'xaxis': {'title': {'text': 'Year'}},
               'yaxis': {'title': {'text': 'Mass in Mg'}}}
})

# Section 5: Export Results

In [29]:
# ====================================================================
# Section 5: Export Results
# ====================================================================

def export_results_to_excel(mfa_system_results, output_filename="mfa_results.xlsx"):
    """
    Exports all calculated flows and stocks into a single Excel file with multiple sheets.
    """
    print(f"\n--> Exporting results to '{output_filename}'...")
    
    time_index = mfa_system_results.IndexTable.Classification['Time'].Items
    elements = mfa_system_results.Elements
    
    with pd.ExcelWriter(output_filename) as writer:
        # --- Export Flows ---
        flow_data_rows = []
        for name, flow_obj in mfa_system_results.FlowDict.items():
            for i, year in enumerate(time_index):
                row = {'Flow_ID': name, 'Year': year}
                for j, element in enumerate(elements):
                    row[element] = flow_obj.Values[i, j]
                flow_data_rows.append(row)
        df_flows = pd.DataFrame(flow_data_rows)
        df_flows.to_excel(writer, sheet_name='Flows_ts', index=False)
        
        # --- Export Stocks ---
        stock_data_rows = []
        for name, stock_obj in mfa_system_results.StockDict.items():
            for i, year in enumerate(time_index):
                row = {'Stock_ID': name, 'Year': year}
                for j, element in enumerate(elements):
                    row[element] = stock_obj.Values[i, j]
                stock_data_rows.append(row)
        df_stocks = pd.DataFrame(stock_data_rows)
        df_stocks.to_excel(writer, sheet_name='Stocks_ts', index=False)
        
    print("--> Export complete.")

In [30]:
# ====================================================================
# Section 5: Export Results
# ====================================================================

# Call the export function
export_results_to_excel(my_mfa_system_with_results, output_filename="rye_mfa_results_v1.xlsx")


--> Exporting results to 'rye_mfa_results_v1.xlsx'...
--> Export complete.


<img src="system_flow_diagram.svg" alt="system_flow_diagram">

## 0 Load packages

This cell imports all necessary packages. It also loads the ODYM framework and the bioDYM_addon, both are included as files in the project (see folder /framework). 

## 3 MFA Calculations

Now, the solution of the MFA is calculated. Since most flows have either input data or TCs and substance contents are given, they can be easily calculated. However, this system includes a dynamic stock modeling (dsm) and a first order model process (FOMP) for the mineralization of carbon in soil. The idea is that first, all flows are calculated with TCs that are independent of dsm or FOMP. Then, dsm is performed and subsequently, all flows up to the FOMP are calculated. After that, the bioDYM_addon functions are used to calculate the mineralization. Finally, all following flows and stocks are calculated.

### 3.1 Solution MFA

### 3.1.1 Solution MFA pt. I (until MBC dynamic stock modelling)


### 3.1.4 Solution MFA pt. IV (FOMP mineralization process)

The carbon mineralization process in the soil is calculated with a first order decay model according to (Cayuela et al., 2010) based on (Robertson & Paul, 2000): 

$$ C_{remaining} (t)=f \cdot exp⁡(-k_{1} \cdot t)+(100\%-f) \cdot exp⁡(-k_{2} \cdot t) $$

To keep calculations simple, it is assumed that this is the only equation that leads to outflows of the process, the remaining fractions of the material accumulate as stock without any emissions. (Cayuela et al., 2010) give parameter values for green waste biochar, they are used here.

\
\
Literature

Cayuela, M. L., Oenema, O., Kuikman, P. J., Bakker, R. R., & Van GROENIGEN, J. W. (2010). Bioenergy by-products as soil amendments? Implications for carbon sequestration and greenhouse gas emissions: C AND N DYNAMICS FROM BIOENERGY BY-PRODUCTS IN SOIL. GCB Bioenergy, no-no. https://doi.org/10.1111/j.1757-1707.2010.01055.x

Robertson, G. P., & Paul, E. (2000). Decomposition and Soil Organic Matter Dynamics. Decomposition and Soil Organic Matter Dynamics., 104–116. https://doi.org/10.1007/978-1-4612-1224-9_8